In [15]:
import sys
import os
sys.path.append(os.path.abspath('../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import StratifiedGroupKFold

from functions.preproc import *
from functions.repeated_cv import *

from sklearn import set_config
set_config(display="text")  # displays text representation of estimators

In [16]:
dataset = pd.read_csv("../data/data.csv")

diagnosis_variables = []

with open("../data/diagnosis.txt", "r") as file:
    for line in file:
        diagnosis_variables.append(line.strip())

survival_dataset = dataset.drop('Unnamed: 0', axis=1)

feature_columns = diagnosis_variables
continuous_features = ['Age', 'Ki67 positive cells (%) at CB', 'N (Regional nodes affected)']

In [17]:
data_x = survival_dataset[feature_columns]
data_y = survival_dataset['Neoadjuvant Response']

In [18]:
groups=survival_dataset['PERSON_ID']

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=12)
cv = sgkf.split(X=data_x, y=survival_dataset['Neoadjuvant Response'], groups=groups)

train_index, test_index = next(cv)
x_train = data_x.iloc[train_index]
y_train = data_y[train_index]

x_test = data_x.iloc[test_index]
y_test = data_y[test_index]

cross = survival_dataset.iloc[train_index]
groups = cross['PERSON_ID']

In [19]:
from sklearn.linear_model import LogisticRegression

model_class = LogisticRegression

param_grid = {
    'penalty': ['elasticnet'],
    'solver': ['saga'],
    'l1_ratio': [0.0, 0.5, 1.0],
    'max_iter': [1000]
}

metrics = ['auc', 'pr-auc', 'accuracy', 'precision', 'recall', 'f1', 'brier']

In [22]:
from sklearn.ensemble import RandomForestClassifier

model_class = RandomForestClassifier

param_grid = {
'n_estimators': [100, 200, 300],
'max_depth': [4, 6, 8],
'max_features': ['sqrt', 'log2'],
'min_samples_leaf': [2, 10, 15, 20],
'bootstrap': [True],
'class_weight': ['balanced', None]
}

metrics = ['auc', 'pr-auc', 'accuracy', 'precision', 'recall', 'f1', 'brier']

In [49]:
from catboost import CatBoostClassifier, Pool

cat_features = [col for col in feature_columns if col not in continuous_features]
cat_features_indices = [data_x.columns.get_loc(col) for col in cat_features]
for col in cat_features:
    x_train[col] = x_train[col].astype(str)

model_class = CatBoostClassifier

param_grid = {
    'iterations': [250, 500],          # relying on early stopping anyway
    'learning_rate': [0.01, 0.05, 0.1], # small steps, signal here is narrow
    'depth': [1, 2, 3],                 # shallow worked before, keeping it in range
    'l2_leaf_reg': [3, 5, 10],
    'random_strength': [1, 5],          # trying to shake it off the plateau
    'bagging_temperature': [0, 1],
    'border_count': [32, 128],
    'verbose': [False] # bin count for the continuous feats
}

metrics = ['auc', 'pr-auc', 'accuracy', 'precision', 'recall', 'f1', 'brier']

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [24]:
results_df, summary_df, best_params = ensemble_repeated_grid_search(model_class, x_train, y_train, param_grid, metrics, #scale for the lr and svm
                                       scale_data=False, scaler_type='standard',
                                      continuous_features=None,
                                       random_state=42, n_jobs=-1,
                                       groups=groups,
                                       n_splits=4, n_repeats=1, base_random_state=42,
                                       verbose=True)


MODEL REPEATED GRID SEARCH
Scaling: No
Grouping: Yes (StratifiedGroupKFold)
Metrics: ['auc', 'pr-auc', 'accuracy', 'precision', 'recall', 'f1', 'brier']
Parameter Grid:
  - n_estimators: [100, 200, 300]
  - max_depth: [4, 6, 8]
  - max_features: ['sqrt', 'log2']
  - min_samples_leaf: [2, 10, 15, 20]
  - bootstrap: [True]
  - class_weight: ['balanced', None]
Total hyperparameter combinations: 144
Folds per repetition: 4
Repetitions: 1
Total evaluations: 576

[1/144] Testing params: {'n_estimators': 100, 'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'bootstrap': True, 'class_weight': 'balanced'}
  auc: 0.7185 ± 0.0296
  pr-auc: 0.8513 ± 0.0242
  accuracy: 0.6421 ± 0.0496
  precision: 0.8243 ± 0.0426
  recall: 0.6125 ± 0.0759
  f1: 0.7009 ± 0.0568
  brier: 0.2129 ± 0.0125
[2/144] Testing params: {'n_estimators': 100, 'max_depth': 4, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'bootstrap': True, 'class_weight': None}
  Elapsed: 0.1min, Est. remaining: 9.5min
  auc: 0.7

In [29]:
summary_df.to_csv('lr_new.csv', index=False)